In [2]:
# 새 커널/다른 ipynb 독립 실행용 코드입니다.
# 이 셀 하나만 실행한 뒤 `generate_ranked_paraphrases_standalone(...)`을 호출하면 됩니다.

import gc
import importlib.util
import re
from pathlib import Path

import numpy as np
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)


def check_standalone_dependencies():
    """Qwen LoRA/QLoRA 추론에 필요한 패키지 설치 여부를 확인합니다."""
    missing_packages = [
        package_name
        for package_name in ["peft", "bitsandbytes"]
        if importlib.util.find_spec(package_name) is None
    ]
    if missing_packages:
        raise ModuleNotFoundError(
            "다음 패키지가 필요합니다: "
            + ", ".join(missing_packages)
            + "\n터미널에서 `uv pip install peft bitsandbytes`를 실행한 뒤 커널을 재시작하세요."
        )


check_standalone_dependencies()
from peft import PeftModel


QWEN_BASE_MODEL_NAME = "Qwen/Qwen3.5-4B-Base"
STS_SCORER_TOKENIZER_NAME = "klue/roberta-base"
QWEN_ADAPTER_DIR_NAME = "qwen3_5_4b_klue_sts_paraphrase_adapter"


def find_project_dir():
    """현재 작업 위치가 프로젝트 루트이든 하위 폴더이든 저장 파일이 있는 폴더를 찾습니다."""
    current = Path.cwd().resolve()
    candidates = [
        current,
        current / "huggingface_transformers_project",
        current.parent,
        current.parent / "huggingface_transformers_project",
    ]

    for candidate in candidates:
        if (candidate / QWEN_ADAPTER_DIR_NAME).exists():
            return candidate

    raise FileNotFoundError(
        "Qwen LoRA adapter 폴더를 찾지 못했습니다. `PROJECT_DIR`을 직접 지정하세요.\n"
        f"찾는 폴더명: {QWEN_ADAPTER_DIR_NAME}"
    )


PROJECT_DIR = find_project_dir()
QWEN_ADAPTER_DIR = PROJECT_DIR / QWEN_ADAPTER_DIR_NAME


def find_sts_scorer_model_dir(project_dir=PROJECT_DIR):
    """v2에서 저장된 STS regression best_model 폴더를 찾습니다."""
    candidates = list(project_dir.glob("transformers_klue_sts_regression_roberta_lr_*/best_model"))
    candidates = [path for path in candidates if (path / "config.json").exists()]

    if not candidates:
        raise FileNotFoundError(
            "STS scorer best_model 폴더를 찾지 못했습니다. `STS_SCORER_MODEL_DIR`을 직접 지정하세요."
        )

    return max(candidates, key=lambda path: path.stat().st_mtime)


STS_SCORER_MODEL_DIR = find_sts_scorer_model_dir()

print("PROJECT_DIR:", PROJECT_DIR)
print("Qwen LoRA adapter:", QWEN_ADAPTER_DIR)
print("STS scorer:", STS_SCORER_MODEL_DIR)


def build_paraphrase_prompt_standalone(sentence):
    """SFT 때 사용한 prompt 형식과 같은 형태로 입력 문장을 감쌉니다."""
    return (
        "### 지시\n"
        "다음 한국어 문장을 의미는 유지하면서 다른 표현의 한 문장으로 바꿔 쓰세요.\n"
        "출력은 paraphrase 문장 하나만 작성하세요.\n\n"
        "### 입력\n"
        f"{sentence}\n\n"
        "### 출력\n"
    )


def load_qwen_lora_for_generation(
    base_model_name=QWEN_BASE_MODEL_NAME,
    adapter_dir=QWEN_ADAPTER_DIR,
):
    """base Qwen 모델을 4-bit로 로드하고 저장된 LoRA adapter를 얹어 생성 모델을 만듭니다."""
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=compute_dtype,
        trust_remote_code=True,
    )
    base_model.config.use_cache = True
    base_model.config.pad_token_id = tokenizer.pad_token_id

    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()
    return tokenizer, model


def load_sts_scorer_standalone(
    scorer_model_dir=STS_SCORER_MODEL_DIR,
    scorer_tokenizer_name=STS_SCORER_TOKENIZER_NAME,
):
    """저장된 STS regression scorer와 tokenizer를 로드합니다."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(scorer_tokenizer_name)
    model = AutoModelForSequenceClassification.from_pretrained(scorer_model_dir).to(device)
    model.eval()
    return tokenizer, model, device


qwen_tokenizer_standalone = None
qwen_model_standalone = None
sts_tokenizer_standalone = None
sts_model_standalone = None
sts_device_standalone = None


def ensure_standalone_models_loaded():
    """필요한 모델이 메모리에 없으면 저장 경로에서 다시 불러옵니다."""
    global qwen_tokenizer_standalone
    global qwen_model_standalone
    global sts_tokenizer_standalone
    global sts_model_standalone
    global sts_device_standalone

    if qwen_model_standalone is None or qwen_tokenizer_standalone is None:
        qwen_tokenizer_standalone, qwen_model_standalone = load_qwen_lora_for_generation()

    if sts_model_standalone is None or sts_tokenizer_standalone is None:
        (
            sts_tokenizer_standalone,
            sts_model_standalone,
            sts_device_standalone,
        ) = load_sts_scorer_standalone()

    return (
        qwen_tokenizer_standalone,
        qwen_model_standalone,
        sts_tokenizer_standalone,
        sts_model_standalone,
        sts_device_standalone,
    )


def clean_paraphrase_output_standalone(text, eos_token=None):
    """생성 결과에서 prompt 조각이나 불필요한 줄을 정리합니다."""
    text = text.strip()
    if eos_token:
        text = text.replace(eos_token, "").strip()
    text = re.split(r"\n###|###|\n입력:|\n지시:", text)[0].strip()
    lines = [line.strip(" -•\t") for line in text.splitlines() if line.strip()]
    return lines[0] if lines else text


def generate_candidates_standalone(
    sentence,
    num_candidates=30,
    batch_return_sequences=5,
    max_new_tokens=80,
    temperature=0.85,
    top_p=0.92,
    repetition_penalty=1.05,
):
    """입력 문장 하나에서 paraphrase 후보를 생성합니다."""
    qwen_tokenizer, qwen_model, _, _, _ = ensure_standalone_models_loaded()
    prompt = build_paraphrase_prompt_standalone(sentence)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(qwen_model.device)

    candidates = []
    seen = set()
    attempts = 0
    max_attempts = max(10, num_candidates * 2)

    while len(candidates) < num_candidates and attempts < max_attempts:
        attempts += 1
        current_batch = min(batch_return_sequences, num_candidates - len(candidates))

        with torch.no_grad():
            outputs = qwen_model.generate(
                **inputs,
                do_sample=True,
                temperature=temperature,
                top_p=top_p,
                repetition_penalty=repetition_penalty,
                max_new_tokens=max_new_tokens,
                num_return_sequences=current_batch,
                pad_token_id=qwen_tokenizer.pad_token_id,
                eos_token_id=qwen_tokenizer.eos_token_id,
            )

        input_length = inputs["input_ids"].shape[-1]
        for output in outputs:
            generated = qwen_tokenizer.decode(output[input_length:], skip_special_tokens=True)
            candidate = clean_paraphrase_output_standalone(generated, qwen_tokenizer.eos_token)
            if candidate and candidate != sentence and candidate not in seen:
                seen.add(candidate)
                candidates.append(candidate)

    return candidates[:num_candidates]


def score_candidates_standalone(source_sentence, candidates, batch_size=32, max_length=128):
    """기존 STS regression scorer로 후보 문장들의 유사도 점수를 계산합니다."""
    _, _, sts_tokenizer, sts_model, sts_device = ensure_standalone_models_loaded()
    scored = []

    for start in range(0, len(candidates), batch_size):
        batch_candidates = candidates[start : start + batch_size]
        encoded = sts_tokenizer(
            [source_sentence] * len(batch_candidates),
            batch_candidates,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_token_type_ids=False,
            return_tensors="pt",
        )
        encoded = {key: value.to(sts_device) for key, value in encoded.items()}

        with torch.no_grad():
            outputs = sts_model(**encoded)

        scores = outputs.logits.squeeze(-1).detach().cpu().numpy()
        scores = np.clip(scores, 0.0, 5.0)

        for candidate, score in zip(batch_candidates, scores):
            scored.append({"candidate": candidate, "score": float(score)})

    return scored


def generate_ranked_paraphrases_standalone(sentence, num_candidates=30, top_k=10):
    """후보 30개를 생성하고 STS 점수가 높은 순으로 top 10을 반환합니다."""
    candidates = generate_candidates_standalone(sentence, num_candidates=num_candidates)
    scored_candidates = score_candidates_standalone(sentence, candidates)
    return sorted(scored_candidates, key=lambda row: row["score"], reverse=True)[:top_k]


def print_ranked_paraphrases_standalone(ranked_results):
    """정렬된 paraphrase 결과를 보기 좋게 출력합니다."""
    for rank, row in enumerate(ranked_results, start=1):
        print(f"{rank:02d}. score={row['score']:.4f} | {row['candidate']}")


def clear_standalone_models():
    """독립 실행 셀에서 로드한 모델을 메모리에서 정리합니다."""
    global qwen_tokenizer_standalone
    global qwen_model_standalone
    global sts_tokenizer_standalone
    global sts_model_standalone
    global sts_device_standalone

    qwen_tokenizer_standalone = None
    qwen_model_standalone = None
    sts_tokenizer_standalone = None
    sts_model_standalone = None
    sts_device_standalone = None

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# 사용 예시:
# sentence = "박대건님의 달콤 살벌한 이야기를 들으려면 제주도에 와야 한다."
# ranked = generate_ranked_paraphrases_standalone(sentence, num_candidates=30, top_k=10)
# print_ranked_paraphrases_standalone(ranked)
print("독립 실행용 함수 준비 완료")

PROJECT_DIR: F:\code\AIFFEL\AIFFEL_study\huggingface_transformers_project
Qwen LoRA adapter: F:\code\AIFFEL\AIFFEL_study\huggingface_transformers_project\qwen3_5_4b_klue_sts_paraphrase_adapter
STS scorer: F:\code\AIFFEL\AIFFEL_study\huggingface_transformers_project\transformers_klue_sts_regression_roberta_lr_5em05\best_model
독립 실행용 함수 준비 완료


In [5]:
# 사용 예시: 아래 문장만 바꿔서 실행하면 됩니다.
input_sentence = "와 이건 정말 대단한 기술이네요! 폴란드에 이미 수출했고, 인도네시아 필리핀 독일 호주 이제 일본에서까지 사고싶어 해요! 대한민국 방산 기술 대단하네요!"

ranked_results = generate_ranked_paraphrases_standalone(
    sentence=input_sentence,
    num_candidates=30,
    top_k=10,
)

print("입력 문장:", input_sentence)
print()
print_ranked_paraphrases_standalone(ranked_results)

입력 문장: 와 이건 정말 대단한 기술이네요! 폴란드에 이미 수출했고, 인도네시아 필리핀 독일 호주 이제 일본에서까지 사고싶어 해요! 대한민국 방산 기술 대단하네요!

01. score=4.3221 | 와, 이것은 정말 대단한 기술입니다! 우리는 이미 폴란드에 그것을 팔았고, 이제 인도네시아, 필리핀, 독일, 호주, 그리고 일본에서 그것을 사고 싶습니다! 한국의 방위 기술은 정말 대단해요!
02. score=4.3060 | 와, 이걸 정말 대단한 기술이라고 생각해요! 폴란드로 이미 수출을 하고 있고, 인도네시아와 필리핀, 독일, 호주를 포함하여 이제는 일본에서도 사고 싶어요! 한국의 방위기술이 너무 훌륭해요!
03. score=4.2907 | 이것은 정말 놀라운 기술입니다! 우리는 이미 폴란드로 수출했으며, 인도네시아, 필리핀, 독일, 호주, 그리고 일본을 사고 싶어요! 한국의 방산 기술이 훌륭합니다!
04. score=4.2792 | 이것은 정말 대단한 기술이에요! 우리는 이미 폴란드로 수출하고 있고, 인도네시아, 필리핀, 독일, 호주 그리고 이제 일본에서 그것을 사고 싶어요! 한국의 무기 기술은 대단합니다!
05. score=4.2780 | 와 이것은 정말 놀라운 기술이에요! 폴란드로부터 이미 수출되었고, 인도네시아, 필리핀, 독일, 호주에서 지금 일본으로 팔고 싶어요! 한국의 방위 기술은 정말 놀랍습니다!
06. score=4.2758 | 폴란드로부터 이미 수출을 시작했고, 인도네시아, 필리핀, 독일, 호주, 그리고 이제 일본까지 사고 싶어요! 대한민국의 군사 기술은 정말 대단합니다!
07. score=4.2737 | 폴란드로 이미 수출한 이 기술은 정말 대단합니다! 인도네시아, 필리핀, 독일, 호주 그리고 이제 일본에서도 판매하고 싶어요! 한국의 방위기술은 대단해요!
08. score=4.2719 | 이것은 정말 놀라운 기술입니다! 폴란드에 이미 판매되어 있고, 인도네시아, 필리핀, 독일, 호주, 그리고 이제 일본에 팔고 싶어요! 대한민국의 방산 기술은

In [6]:
# 사용 예시: 아래 문장만 바꿔서 실행하면 됩니다.
input_sentence = "떡볶이는 식감때문에 외국인들이 처음에 거부감을 보이지만 한번 맛보면 중독된다고 하더라고요."

ranked_results = generate_ranked_paraphrases_standalone(
    sentence=input_sentence,
    num_candidates=30,
    top_k=10,
)

print("입력 문장:", input_sentence)
print()
print_ranked_paraphrases_standalone(ranked_results)

입력 문장: 떡볶이는 식감때문에 외국인들이 처음에 거부감을 보이지만 한번 맛보면 중독된다고 하더라고요.

01. score=4.5441 | 외국인들은 처음에는 식감 때문에 떡볶이에 거부감을 느끼지만, 한번 맛보면 중독될 수 있다고 합니다.
02. score=4.4959 | 떡볶이는 식감이기 때문에 외국인들은 처음에 거부감을 보이는데, 한번 맛을 보면 중독이 된다고 합니다.
03. score=4.4880 | 외국인들은 처음 떡볶이의 식감 때문에 거부감을 보이지만, 한번 맛을 보면 중독될 거라고 합니다.
04. score=4.3787 | 떡볶이는 식감 때문에 외국인들이 처음에는 거부감을 보이지만, 한 번 맛보면 중독될 거예요.
05. score=4.3719 | 떡볶이는 식감 때문에 외국인들은 처음에는 거부감을 느끼지만, 한번 맛보면 중독되는 것이라고 들었습니다.
06. score=4.3657 | 떡볶이는 식감이기 때문에 외국인들은 처음에는 거부감을 보이지만, 한 번 맛보면 중독됩니다.
07. score=4.3657 | 외국인들은 떡볶이의 식감 때문에 처음 거부감을 보이지만, 한번 맛을 보니 중독되어요.
08. score=4.3476 | 외국인들은 처음에는 식감 때문에 떡볶이에 거부감을 가지지만, 한번 먹으면 중독될 거라고 들었어요.
09. score=4.3158 | 식감 때문에 외국인들은 떡볶이에 처음 거부감을 보이지만, 한 번 맛보면 중독될 수 있습니다.
10. score=4.2784 | 떡볶이는 식감 때문에 외국인들은 처음에는 거부감을 보이지만, 일단 맛을 보면 중독되는 것 같아요.
